<a href="https://colab.research.google.com/github/thorayya/online_exam_cheating_detection/blob/main/object_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics
!pip install -q mediapipe
!pip install boxmot

In [ ]:
from ultralytics import YOLO
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from boxmot.trackers import OccluBoost
import cv2
import csv
import numpy as np

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from google.colab import files
files.upload()

In [ ]:
!wget -O pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

In [ ]:
# Create an PoseLandmarker object.
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
PoseLandmarkerResult = mp.tasks.vision.PoseLandmarkerResult
VisionRunningMode = mp.tasks.vision.RunningMode

model_path = "pose_landmarker.task"


cap = cv2.VideoCapture("video.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
frame_id = 0

classes = ["cell phone", "book", "clock", "laptop"]


def load_models():
  model = YOLO("yolov8s.pt")

  options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO)

  tracker = OccluBoost()

  landmarker = PoseLandmarker.create_from_options(options)

  return model, landmarker, tracker


model, landmarker, tracker = load_models()

def object_detection(frame):


  results = model(frame)
  objects_boxes = []
  dets = []

  for result in results:
      boxes = result.boxes

      for box in boxes:
        x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()  # Get the coordinates
        conf = box.conf[0].cpu().numpy() # Confidence score
        cls = int(box.cls[0].cpu().numpy())# Class ID

        name = model.names[cls]

        if name in classes:
          x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()  # Get the coordinates

          dets.append([x1,y1,x2,y2,conf,cls])

          center_x = (x1 + x2)/ 2
          center_y =(y1 + y2)/2

          objects_boxes.append((center_x,center_y))

  return objects_boxes, dets




def pose_detection(frame_rgb,timestamp_ms, w, h):

  result = landmarker.detect_for_video(frame_rgb, timestamp_ms)


  if not result.pose_landmarks:
    return None

  else:

    landmark = result.pose_landmarks[0]


    right_wrist = landmark[16]
    left_wrist = landmark[15]

    left_shoulder = landmark[11]
    right_shoulder = landmark[12]

    right_elbow = landmark[14]
    left_elbow = landmark[13]


    x_right_wrist = int(right_wrist.x * w)
    y_right_wrist = int(right_wrist.y * h)

    x_left_wrist = int(left_wrist.x * w)
    y_left_wrist = int(left_wrist.y * h)

    x_left_shoulder = int(left_shoulder.x * w)
    y_left_shoulder = int(left_shoulder.y * h)

    x_right_shoulder = int(right_shoulder.x * w)
    y_right_shoulder = int(right_shoulder.y * h)

    x_right_elbow = int(right_elbow.x * w)
    y_right_elbow = int(right_elbow.y * h)

    x_left_elbow = int(left_elbow.x * w)
    y_left_elbow = int(left_elbow.y * h)

    position={
          "right_wrist": (x_right_wrist, y_right_wrist),
          "left_wrist": (x_left_wrist, y_left_wrist),
          "left_shoulder": (x_left_shoulder, y_left_shoulder),
          "right_shoulder": (x_right_shoulder, y_right_shoulder),
          "right_elbow": (x_right_elbow, y_right_elbow),
          "left_elbow": (x_left_elbow, y_left_elbow)
      }

  return position





def tracking(dets, frame):

  if len(dets) > 0:
    dets = np.array(dets, dtype=np.float32)
  else:
    dets = np.empty((0, 6), dtype=np.float32)

  tracks = tracker.update(dets, frame)

  return tracks


def get_nearest_object(pose, objects_boxes):

  if pose is None or len(objects_boxes) == 0:
    return None

  else:
    distances = []

    for box in objects_boxes:
      d1 = np.linalg.norm(np.array(pose["left_wrist"]) - np.array(box))
      d2 = np.linalg.norm(np.array(pose["right_wrist"]) - np.array(box))

      distances.append(min(d1,d2))

    nearest_object = np.argmin(distances)

    return objects_boxes[nearest_object]



def build_features(pose, tracks, objects_boxes):

  if pose is None or len(objects_boxes) == 0:
    return None

  else:
    obj = get_nearest_object(pose, objects_boxes)
    if obj is None:
      return None


    left_wrist_distance_to_object = np.linalg.norm(np.array(pose["left_wrist"]) - np.array(obj))
    right_wrist_distance_to_object = np.linalg.norm(np.array(pose["right_wrist"]) - np.array(obj))

    left_elbow_distance_to_object = np.linalg.norm(np.array(pose["left_elbow"])- np.array(obj))
    right_elbow_distance_to_object = np.linalg.norm(np.array(pose["right_elbow"]) - np.array(obj))

    return (
            left_wrist_distance_to_object,
            right_wrist_distance_to_object,
            left_elbow_distance_to_object,
            right_elbow_distance_to_object
            )


def angle(pose_shoulder, pose_elbow, pose_wrist):

  pose_shoulder = np.array(pose_shoulder)
  pose_elbow = np.array(pose_elbow)
  pose_wrist = np.array(pose_wrist)


  elbow_shoulder = pose_shoulder - pose_elbow
  wrist_elbow = pose_wrist - pose_elbow

  cosine = np.dot(elbow_shoulder, wrist_elbow) / (np.linalg.norm(elbow_shoulder) * np.linalg.norm(wrist_elbow) + 1e-6)

  return np.degrees(np.arccos(cosine))





# with open("data.csv", "w", newline="") as f:
#   writer = csv.writer(f)
#   writer.writerow(["cell phone", "book", "clock", "laptop"])




while cap.isOpened():
  ret, frame = cap.read()


  if not ret:
    print("Failed to grab frame")
    break


  # ---------------- YOLO ----------------

  objects_boxes, dets = object_detection(frame)


  # ---------------- POSE ----------------



  h, w, _ = frame.shape


  frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
  frame_rgb = mp.Image(image_format=mp.ImageFormat.SRGB,data=frame_rgb)




  # ---------------- TRACKING ----------------

  try:

    tracks = tracking(dets, frame)
  except:
    tracks = []


  timestamp_ms = int((frame_id / fps) * 1000)

  pose = pose_detection(frame_rgb, timestamp_ms, w, h)



  if pose is not None:

    features = build_features(pose, tracks, objects_boxes)

    left_elbow_angle = angle(pose["left_shoulder"],
                              pose["left_elbow"],
                              pose["left_wrist"])

    right_elbow_angle = angle(pose["right_shoulder"],
                               pose["right_elbow"],
                               pose["right_wrist"])

    if features is not None:
      print(f"FEATURES:", features)



  frame_id += 1



  # cv2.imshow("title", frame)

  if cv2.waitKey(1) & 0xFF == ord('q'):
    break



cv2.destroyAllWindows()
cap.release()




INFO     BoostTrack: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,       
         asso_func=iou, reid_model=None, use_ecc=True, min_box_area=10, aspect_ratio_thresh=1.6, cmc_method=ecc,   
         lambda_iou=0.5, lambda_mhd=0.25, lambda_shape=0.25, use_dlo_boost=True, use_duo_boost=True,               
         dlo_boost_coef=0.65, s_sim_corr=False, use_rich_s=False, use_sb=False, use_vt=False, with_reid=False,     
         adaptive_kf=False


0: 384x640 3 persons, 1 cell phone, 2705.8ms
Speed: 17.8ms preprocess, 2705.8ms inference, 71.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 cell phone, 716.0ms
Speed: 5.6ms preprocess, 716.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 1 cell phone, 371.1ms
Speed: 3.0ms preprocess, 371.1ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 1 cell phone, 386.0ms
Speed: 2.7ms preprocess, 386.0ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 1 cell phone, 377.1ms
Speed: 2.6ms preprocess, 377.1ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 1 cell phone, 413.7ms
Speed: 2.6ms preprocess, 413.7ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 1 cell phone, 373.3ms
Speed: 2.6ms preprocess, 373.3ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 